# Customer Churn Intelligence — Data Cleaning

## Objective

Apply deterministic data-quality fixes identified in `01_eda.ipynb` and save a cleaned dataset to `data/processed/`.

**Stage:** Step 4 — Data Cleaning (no train/test split, no encoding pipelines, no modeling).

## Issues to Address (from EDA)

| Issue | Action in this step |
|-------|---------------------|
| `TotalCharges` stored as `object` with 11 blank strings | Coerce to `float64`; impute `0.0` where `tenure = 0` |
| `customerID` is an identifier | Retain for linkage; exclude from modeling later |
| Sentinel categories (`No internet service`, etc.) | Defer to preprocessing/encoding stage |
| `SeniorCitizen` already 0/1 | No change needed |
| No duplicate rows or explicit NaNs | Verify only |

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import (
    CLEANED_FILENAME,
    RAW_FILENAME,
    load_raw_data,
    run_cleaning_pipeline,
    validate_cleaned_data,
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / RAW_FILENAME
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / CLEANED_FILENAME

## 1. Load Raw Data (unchanged source)

In [ ]:
raw_df = load_raw_data(RAW_PATH)
print(f"Raw shape: {raw_df.shape}")
print(f"TotalCharges dtype: {raw_df['TotalCharges'].dtype}")

blank_total_charges = raw_df["TotalCharges"].astype(str).str.strip().eq("").sum()
print(f"Blank TotalCharges rows: {blank_total_charges}")

## 2. Apply Cleaning

In [ ]:
cleaned_df, cleaning_audit, validation = run_cleaning_pipeline(
    raw_path=RAW_PATH,
    processed_path=PROCESSED_PATH,
)

print(f"Saved cleaned data to: {cleaning_audit['output_path']}")
print(f"Rows imputed for TotalCharges: {cleaning_audit['total_charges_imputed_rows']}")
display(cleaning_audit["total_charges_imputation_audit"])

## 3. Before vs After

In [ ]:
comparison = pd.DataFrame(
    {
        "Metric": [
            "Rows",
            "Columns",
            "TotalCharges dtype (raw)",
            "TotalCharges dtype (cleaned)",
            "Blank/non-numeric TotalCharges (raw)",
            "Missing values (cleaned)",
            "Duplicate rows (cleaned)",
        ],
        "Value": [
            cleaned_df.shape[0],
            cleaned_df.shape[1],
            str(raw_df["TotalCharges"].dtype),
            str(cleaned_df["TotalCharges"].dtype),
            int(pd.to_numeric(raw_df["TotalCharges"], errors="coerce").isna().sum()),
            int(cleaned_df.isnull().sum().sum()),
            int(cleaned_df.duplicated().sum()),
        ],
    }
)
comparison

In [ ]:
print("Churn distribution — raw:")
display(raw_df["Churn"].value_counts())
print("\nChurn distribution — cleaned (should match):")
display(cleaned_df["Churn"].value_counts())

In [ ]:
print("Cleaned dtypes:")
cleaned_df.dtypes

In [ ]:
print("Validation checks:")
validation

## Data Cleaning Summary

Facts observed after cleaning:

- **Rows preserved:** 7,043 (no rows dropped).
- **Columns preserved:** 21 (no columns added or removed).
- **`TotalCharges`:** Coerced from `object` to `float64`. Eleven blank values (all `tenure = 0`) imputed to `0.0` because new customers have not yet accumulated charges.
- **Target distribution:** Unchanged — No: 5,174; Yes: 1,869.
- **Missing values:** 0 across all columns after cleaning.
- **`customerID`:** Retained for prediction linkage; will be excluded from features during modeling.
- **Sentinel categories:** Left unchanged; encoding deferred to preprocessing.
- **Output file:** `data/processed/cleaned_churn.csv`

**Next step (not performed here):** Leakage audit and train/validation/test split.